### Fruit Recommendation System using Content-Based Filtering and Cosine Similarity


# Problem Statement

People often want to choose foods with similar nutritional values for:

    healthy diet planning
    
    nutritional substitution
    
    meal recommendations

However, it is difficult to manually compare the nutritional values of many fruits.

Therefore, we build a recommendation system that automatically finds nutritionally similar fruits.


| Column      | Description    |
| ----------- | -------------- |
| fruit_100g  | Name of fruit  |
| energy_kcal | Energy content |
| water_g     | Water content  |
| protein_g   | Protein        |
| fat_g       | Fat            |
| fiber_g     | Dietary fiber  |
| sugars_g    | Sugar content  |
| vitaminc_mg | Vitamin C      |


In [1]:

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:

fruit_nutrition = pd.read_csv("fruit_nutrition.csv")

fruit_nutrition.head()

,fruit_100g,energy_kcal,water_g,protein_g,totalfat_g,fiber_g,sugars_g,vitaminc_mg
0,Banana,89,74.9,1.09,0.33,2.6,12.20,8.7
1,Lemon,29,89.0,1.10,0.30,2.8,2.50,53.0
2,Lime,30,88.3,0.70,0.20,2.8,1.69,29.1
3,Mango,46,88.3,0.91,0.27,1.5,8.39,4.1
4,Peach,60,83.5,0.82,0.38,1.6,13.70,36.4


---

### Set Fruit Name as Index

Index makes it easier to reference fruits.

In [3]:

nutrition = fruit_nutrition.set_index("fruit_100g")
nutrition

,energy_kcal,water_g,protein_g,totalfat_g,fiber_g,sugars_g,vitaminc_mg
fruit_100g,,,,,,,
Banana,89,74.9,1.09,0.33,2.6,12.20,8.7
Lemon,29,89.0,1.10,0.30,2.8,2.50,53.0
Lime,30,88.3,0.70,0.20,2.8,1.69,29.1
Mango,46,88.3,0.91,0.27,1.5,8.39,4.1
Peach,60,83.5,0.82,0.38,1.6,13.70,36.4
Pineapple,50,86.0,0.54,0.12,1.4,9.85,47.8


In [4]:

nutrition.index.name = None

nutrition.head()

,energy_kcal,water_g,protein_g,totalfat_g,fiber_g,sugars_g,vitaminc_mg
Banana,89,74.9,1.09,0.33,2.6,12.20,8.7
Lemon,29,89.0,1.10,0.30,2.8,2.50,53.0
Lime,30,88.3,0.70,0.20,2.8,1.69,29.1
Mango,46,88.3,0.91,0.27,1.5,8.39,4.1
Peach,60,83.5,0.82,0.38,1.6,13.70,36.4


In [ ]:

nutrition.loc["Mango"]

### Feature Selection

In Content-Based systems we must choose features that describe items.

In [5]:
features = nutrition[["sugars_g", "vitaminc_mg"]]

features.head()

,sugars_g,vitaminc_mg
Banana,12.20,8.7
Lemon,2.50,53.0
Lime,1.69,29.1
Mango,8.39,4.1
Peach,13.70,36.4



These columns represent fruit characteristics.

Mango  → [8.39 , 4.1]

Banana → [12.2 , 8.7]

### Calculate Similarity Between Two Fruits

In [8]:
cosine_similarity(features.loc[["Mango","Banana"]])[0][1]

0.9864305980007304


Cosine similarity measures how similar two vectors are.

    0  → completely different
    
    1  → identical

Similarity(Mango,Banana) = 0.93

Mango and Banana have similar nutritional composition



### Compute Similarity for All Fruits


Instead of comparing only two fruits, we compare every fruit with every fruit.


In [10]:
import numpy as np

np.set_printoptions(linewidth=np.inf)

In [9]:

similarity_matrix = cosine_similarity(features)

print(similarity_matrix)

[[1.         0.61832414 0.62683477 0.9864306  0.83018982 0.73298243]
 [0.61832414 1.         0.99994086 0.48090193 0.95146351 0.98784307]
 [0.62683477 0.99994086 1.         0.49040872 0.95475426 0.98947527]
 [0.9864306  0.48090193 0.49040872 1.         0.72739811 0.61135407]
 [0.83018982 0.95146351 0.95475426 0.72739811 1.         0.98773952]
 [0.73298243 0.98784307 0.98947527 0.61135407 0.98773952 1.        ]]



Important concept:  Diagonal = 1

                    Because an item is 100% similar to itself.


### Convert Similarity Matrix to DataFrame

In [11]:

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=features.index,
    columns=features.index
)

similarity_df.head()

,Banana,Lemon,Lime,Mango,Peach,Pineapple
Banana,1.000000,0.618324,0.626835,0.986431,0.830190,0.732982
Lemon,0.618324,1.000000,0.999941,0.480902,0.951464,0.987843
Lime,0.626835,0.999941,1.000000,0.490409,0.954754,0.989475
Mango,0.986431,0.480902,0.490409,1.000000,0.727398,0.611354
Peach,0.830190,0.951464,0.954754,0.727398,1.000000,0.987740


### Predict Recommendation

If a person likes Mango, what fruits should we recommend?

In [16]:

similarity_df[["Mango"]].sort_values(by="Mango", ascending=False)[1:3]


,Mango
Banana,0.986431
Peach,0.727398


### Using All Nutritional Features

In [17]:

similarity_matrix_all = cosine_similarity(nutrition)
similarity_matrix_all

array([[1.        , 0.77186848, 0.83412488, 0.92474189, 0.93676783, 0.87931265],
       [0.77186848, 1.        , 0.97739451, 0.87650678, 0.94133956, 0.9780653 ],
       [0.83412488, 0.97739451, 1.        , 0.95282112, 0.95471941, 0.96986632],
       [0.92474189, 0.87650678, 0.95282112, 1.        , 0.94580335, 0.91804609],
       [0.93676783, 0.94133956, 0.95471941, 0.94580335, 1.        , 0.98972148],
       [0.87931265, 0.9780653 , 0.96986632, 0.91804609, 0.98972148, 1.        ]])

In [18]:

similarity_all_df = pd.DataFrame(
    similarity_matrix_all,
    index=nutrition.index,
    columns=nutrition.index
)

similarity_all_df.head()

,Banana,Lemon,Lime,Mango,Peach,Pineapple
Banana,1.000000,0.771868,0.834125,0.924742,0.936768,0.879313
Lemon,0.771868,1.000000,0.977395,0.876507,0.941340,0.978065
Lime,0.834125,0.977395,1.000000,0.952821,0.954719,0.969866
Mango,0.924742,0.876507,0.952821,1.000000,0.945803,0.918046
Peach,0.936768,0.941340,0.954719,0.945803,1.000000,0.989721


### Build Recommendation Function

### To recommend fruits:

    Select a fruit the user likes
    
    Find fruits with highest similarity scores
    
    Return the top recommendations 


In [22]:

def recommend_fruits(fruit_name, similarity_df, top_n=3):
    
    # check if fruit exists
    if fruit_name not in similarity_df.index:
        return f"{fruit_name} not found in dataset"
    
    # sort similarity scores
    similar_fruits = similarity_df[fruit_name].sort_values(ascending=False)
    
    # remove the fruit itself
    recommendations = similar_fruits.iloc[1:top_n+1]
    
    return recommendations
    

In [25]:
recommend_fruits("Lime", similarity_all_df)

Lemon        0.977395
Pineapple    0.969866
Peach        0.954719
Name: Lime, dtype: float64

In [24]:
recommend_fruits("Kiwi", similarity_all_df)

'Kiwi not found in dataset'